# 🥉 Ingestão Batch: Camada Bronze

**Sessão 3.1 do roteiro, Etapa 3, branch `feature/ingestao-batch`**

Este notebook desenvolve a ingestão batch da pipeline: extrai as fontes consolidadas do BigQuery público da Base dos Dados e as grava na camada Bronze do data lake, em formato Parquet, com metadados de ingestão.

> **Convenção de trabalho:** o desenvolvimento acontece neste notebook, célula a célula, com os resultados salvos. Ao final da etapa, o código estável é promovido para `src/ingestion/prod_ingestao_batch.py`, que é a versão executada pela orquestração.

**O que entra por batch** (decisão D-005 do diário): as publicações consolidadas, passadas e futuras: diretório de municípios, metas pactuadas, indicador por UF e município e microdados de alunos (histórico 2023 e 2024).

**Regras da camada Bronze** (README, seção 4.2):
- dado gravado como chegou da fonte, sem regra de negócio;
- metadados de ingestão em cada registro (timestamp e origem);
- formato Parquet, particionado por data de ingestão;
- reexecução não pode duplicar dados (idempotência).

📚 **Referências das aulas (módulo Fase 2, Data Prepare):**
- *ETL Pipelines, Aula 1 (Pipelines ETL/ELT):* arquitetura medalhão, papel da camada Bronze, metadados de ingestão, tipos de extração (full e incremental) e o paradigma ELT;
- *ETL Pipelines, Aula 3 (Cloud Data):* separação entre computação e armazenamento, object storage como fundação do lake, BigQuery;
- *ETL Pipelines, Aula 4 (FinOps):* formato Parquet e particionamento como decisões de custo;
- *Notebook ETL_Pipeline_Demo_Parte1_Pandas:* implementação de referência das camadas Bronze/Silver/Gold vista em aula.

---

**Estratégia de desenvolvimento deste notebook:**

```
uf (menor tabela) -> validar o ciclo completo -> converter em função -> escalar para todas
   145 linhas         ingerir, conferir,           Seção 7               Seções 7 e 8
                      reexecutar (Seções 4 a 6)
```

O ciclo completo de ingestão é validado primeiro na menor tabela, onde um erro custa segundos. Em seguida, o código validado é generalizado como função e executado para as demais tabelas. Ao final da etapa, o código estável é promovido para `src/ingestion/prod_ingestao_batch.py`. Este notebook permanece no repositório como registro do desenvolvimento e material de estudo.

## 1. Setup do ambiente

**Passos desta seção:** importar as bibliotecas e definir as constantes do projeto (identificadores do GCP e da fonte).

🎓 **Conceito** (*ETL Pipelines, Aula 1*): a extração é a etapa E do ELT. Aqui definimos de onde os dados vêm (BigQuery público `basedosdados`) e qual projeto assina as consultas (projeto GCP próprio, dentro do free tier de 1 TB/mês). A credencial foi autorizada na exploração (Etapa 1) e fica armazenada na máquina, por isso nenhuma senha aparece no código.

> 📌 **Nota para reprodução:** a constante `PROJETO_GCP` na célula abaixo identifica o projeto Google Cloud de quem executa (responsável pelo billing das consultas). Quem for reproduzir este projeto deve substituí-la pelo ID do próprio projeto, criado gratuitamente em `console.cloud.google.com/projectcreate`. Na versão de produção (`src/`), esta constante será parametrizada em `config/`.

In [17]:
# --- imports ---
import pandas_gbq                      # ponte pandas <-> BigQuery (roda o SQL e devolve DataFrame)
from datetime import datetime, timezone

# --- constantes do projeto ---
# Projeto GCP que assina as consultas (billing do free tier)
PROJETO_GCP = "tech-challenge-fase2"

# Endereço do dataset público da Base dos Dados no BigQuery (a FONTE)
FONTE = "basedosdados.br_inep_avaliacao_alfabetizacao"

# Tabelas que entram por batch (decisão D-005)
TABELAS_BATCH = [
    "uf",
    "municipio",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "dicionario",
    # "alunos" fica por último: é a maior (3,9 mi de linhas) — entra
    # depois que o fluxo estiver validado com as tabelas pequenas
]

print("Setup ok —", len(TABELAS_BATCH), "tabelas na fila de ingestão")

Setup ok — 6 tabelas na fila de ingestão


## 2. Teste de conexão com a fonte

**Passos desta seção:** executar uma consulta mínima (5 linhas da menor tabela) para confirmar que a credencial e o projeto estão funcionando antes de qualquer ingestão.

🎓 **Conceito** (*ETL Pipelines, Aula 1*): validar a fonte antes de processar é rotina profissional: uma falha de extração detectada no início custa segundos; detectada no meio da carga, custa reprocessamento. É o mesmo princípio da verificação de diretórios (`dbutils.fs.ls()`) utilizada nas aulas de Spark antes de qualquer leitura.

In [18]:
# Consulta mínima: LIMIT 5 lê pouquíssimos bytes (custo ~zero)
df_teste = pandas_gbq.read_gbq(
    f"SELECT * FROM `{FONTE}.uf` LIMIT 5",
    project_id=PROJETO_GCP,
    progress_bar_type=None,
)

print(f"Conexão ok — {len(df_teste)} linhas, {len(df_teste.columns)} colunas")
df_teste  # última expressão da célula aparece como tabela logo abaixo

Conexão ok — 5 linhas, 15 colunas


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,AP,2,3,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,PE,2,5,58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Destino da Bronze: o bucket no Google Cloud Storage

**Passos desta seção:** (3.1) autorizar o acesso ao Storage no navegador; (3.2) criar o bucket do data lake, caso ainda não exista.

🎓 **Conceito** (*ETL Pipelines, Aula 3, Cloud Data*): o object storage é a fundação do data lake: armazenamento durável e de baixo custo, desacoplado da computação. Um bucket é o repositório onde as camadas Bronze, Silver e Gold viverão como arquivos Parquet (decisão D-006: dados fora do Git).

🎓 **Conceito FinOps** (*ETL Pipelines, Aula 4*): a escolha da região é uma decisão de custo. O free tier do GCS (5 GB/mês) vale apenas para regiões dos EUA (`us-central1`, `us-east1`, `us-west1`), e o BigQuery da Base dos Dados também reside nos EUA, o que evita custo de transferência entre regiões (egress). Por isso, `us-central1`.

⚠️ **Sobre a célula 3.1:** a credencial usada até aqui tinha permissão apenas de BigQuery. Para operar o Storage, o navegador abrirá uma vez solicitando uma autorização mais ampla (Google Cloud Platform). Após autorizar, a credencial fica armazenada e não é solicitada novamente.

> 📌 **Nota para reprodução:** a constante `BUCKET_LAKE` (célula 3.2) também é individual: nomes de bucket são únicos globalmente, portanto quem reproduzir deve definir o próprio nome. Na versão de produção (`src/`), esta constante será parametrizada em `config/`.

In [19]:
# --- 3.1 Autorização com escopo ampliado (BigQuery + Storage) ---
# A credencial anterior tinha escopo só de BigQuery. O escopo
# "cloud-platform" cobre BigQuery E Storage com uma credencial só.
# pydata_google_auth guarda a credencial na máquina: o navegador
# abre na primeira execução e nunca mais.
import pydata_google_auth

ESCOPOS = ["https://www.googleapis.com/auth/cloud-platform"]

credenciais = pydata_google_auth.get_user_credentials(ESCOPOS)
print("Credencial ok — válida para BigQuery e Cloud Storage")

Credencial ok — válida para BigQuery e Cloud Storage


In [20]:
# --- 3.2 Criar o bucket do data lake (se não existir) ---
from google.cloud import storage

# Nome do bucket: precisa ser ÚNICO NO MUNDO (por isso o sufixo do RM).
# Pode trocar antes de rodar, se preferir outro nome.
BUCKET_LAKE = "tech-challenge-fase2-lake-rm373453"
REGIAO      = "us-central1"   # free tier + mesma região da fonte (FinOps)

cliente_storage = storage.Client(project=PROJETO_GCP, credentials=credenciais)

# Idempotência desde já: criar só se não existir (rodar 2x não dá erro)
nomes_existentes = [b.name for b in cliente_storage.list_buckets()]
if BUCKET_LAKE in nomes_existentes:
    print(f"Bucket já existe: gs://{BUCKET_LAKE}")
else:
    cliente_storage.create_bucket(BUCKET_LAKE, location=REGIAO)
    print(f"Bucket criado: gs://{BUCKET_LAKE} (região {REGIAO})")

Bucket já existe: gs://tech-challenge-fase2-lake-rm373453


## 4. Primeira ingestão: tabela `uf`

**Passos desta seção:** (4.1) extrair a tabela `uf` completa e adicionar os metadados de ingestão; (4.2) gravar como Parquet particionado no bucket e conferir o resultado no lake, comparando o tamanho com o equivalente em CSV.

🎓 **Conceito, metadados de ingestão** (*ETL Pipelines, Aula 1*): a Bronze guarda o dado como chegou, acrescido de duas colunas de rastreabilidade que a fonte não tem: `_ingestion_timestamp` (quando o dado foi ingerido) e `_source` (de onde veio). São elas que tornam a camada auditável: qualquer número na Gold pode ser rastreado até quando e de onde entrou. O prefixo `_` distingue metadado de dado.

🎓 **Conceito, particionamento por data de ingestão** (*ETL Pipelines, Aulas 1 e 4*): o caminho `bronze/uf/data_ingestao=AAAA-MM-DD/` organiza cada carga em uma partição datada. Isso preserva o histórico de cargas (auditoria), permite reprocessar um dia específico e habilita o partition pruning (ler apenas a partição necessária). Também garante idempotência: reexecutar no mesmo dia sobrescreve a mesma partição, sem duplicar.

🎓 **Conceito, formato Parquet** (*ETL Pipelines, Aula 4, FinOps*): o Parquet é colunar e comprimido: ocupa menos espaço e permite ler somente as colunas necessárias. A célula 4.2 mede a diferença com os dados do projeto.

> 🔎 **Verificação manual (sem Python):** após executar a célula 4.2, o resultado pode ser conferido no console web do Cloud Storage, no endereço `https://console.cloud.google.com/storage/browser/<NOME_DO_SEU_BUCKET>`, substituindo `<NOME_DO_SEU_BUCKET>` pelo valor definido em `BUCKET_LAKE` (célula 3.2). O link funciona apenas para o dono do bucket; cada pessoa acessa o seu. A árvore esperada é `bronze/ > uf/ > data_ingestao=AAAA-MM-DD/ > uf.parquet`. Essa conferência faz parte do processo, e uma captura de tela serve de evidência para a documentação.

In [21]:
# --- 4.1 Extrair a tabela completa + metadados de ingestão ---
# Extração FULL (tabela inteira): adequada aqui porque a fonte é pequena
# e consolidada — a "gaveta" do dia registra a foto completa da carga.
# (ETL Aula 1: full × incremental é decisão por tabela.)

TABELA = "uf"

df_uf = pandas_gbq.read_gbq(
    f"SELECT * FROM `{FONTE}.{TABELA}`",
    project_id=PROJETO_GCP,
    progress_bar_type=None,
)

# Carimbos de rastreabilidade (a fonte não os tem; a Bronze exige):
agora = datetime.now(timezone.utc)
df_uf["_ingestion_timestamp"] = agora.isoformat()          # quando ingerimos
df_uf["_source"]              = f"{FONTE}.{TABELA}"        # de onde veio

print(f"Extraída: {TABELA} — {len(df_uf)} linhas, {len(df_uf.columns)} colunas (15 da fonte + 2 metadados)")
df_uf.head(3)

Extraída: uf — 145 linhas, 17 colunas (15 da fonte + 2 metadados)


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-09T02:40:01.242047+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-09T02:40:01.242047+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-09T02:40:01.242047+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf


In [22]:
# --- 4.2 Gravar Parquet particionado no lake + conferir ---
# Caminho no padrão da Bronze: bronze/<tabela>/data_ingestao=<dia>/
DATA_INGESTAO = agora.strftime("%Y-%m-%d")
caminho = f"gs://{BUCKET_LAKE}/bronze/{TABELA}/data_ingestao={DATA_INGESTAO}/{TABELA}.parquet"

# storage_options passa a credencial ampliada da Seção 3 para o gcsfs
# (a biblioteca que ensina o pandas a escrever em gs://)
df_uf.to_parquet(caminho, index=False, storage_options={"token": credenciais})
print(f"Gravado: {caminho}")
print()

# Conferência 1 — o arquivo existe no lake? Qual o tamanho?
blobs = list(cliente_storage.list_blobs(BUCKET_LAKE, prefix="bronze/"))
for b in blobs:
    print(f"  gs://{BUCKET_LAKE}/{b.name}  ({b.size:,} bytes)")

# Conferência 2 — FinOps na prática: quanto ocupariam os mesmos dados em CSV?
tamanho_csv     = len(df_uf.to_csv(index=False).encode("utf-8"))
tamanho_parquet = blobs[0].size if blobs else 0
print()
if tamanho_parquet:
    razao = tamanho_csv / tamanho_parquet
    print(f"Mesmos dados: CSV = {tamanho_csv:,} bytes | Parquet = {tamanho_parquet:,} bytes")
    if razao >= 1:
        print(f"O Parquet ficou {razao:.1f}x menor que o CSV")
    else:
        # Honestidade estatística: em tabelas minúsculas (145 linhas!) o
        # overhead de metadados do Parquet pode dominar e ele sai MAIOR.
        # A vantagem do formato aparece com volume — veremos na tabela
        # alunos (3,9 mi de linhas), na Seção 8.
        print(f"Curioso: aqui o Parquet ficou {1/razao:.1f}x MAIOR — tabela pequena demais;")
        print("o ganho do formato colunar aparece com volume (aguarde a tabela alunos)")

Gravado: gs://tech-challenge-fase2-lake-rm373453/bronze/uf/data_ingestao=2026-07-09/uf.parquet

  gs://tech-challenge-fase2-lake-rm373453/bronze/alunos/data_ingestao=2026-07-08/alunos.parquet  (70,913,587 bytes)
  gs://tech-challenge-fase2-lake-rm373453/bronze/dicionario/data_ingestao=2026-07-08/dicionario.parquet  (5,197 bytes)
  gs://tech-challenge-fase2-lake-rm373453/bronze/meta_alfabetizacao_brasil/data_ingestao=2026-07-08/meta_alfabetizacao_brasil.parquet  (9,458 bytes)
  gs://tech-challenge-fase2-lake-rm373453/bronze/meta_alfabetizacao_municipio/data_ingestao=2026-07-08/meta_alfabetizacao_municipio.parquet  (222,837 bytes)
  gs://tech-challenge-fase2-lake-rm373453/bronze/meta_alfabetizacao_uf/data_ingestao=2026-07-08/meta_alfabetizacao_uf.parquet  (12,417 bytes)
  gs://tech-challenge-fase2-lake-rm373453/bronze/municipio/data_ingestao=2026-07-08/municipio.parquet  (497,758 bytes)
  gs://tech-challenge-fase2-lake-rm373453/bronze/uf/data_ingestao=2026-07-08/uf.parquet  (17,883 bytes

## 5. Conferência: leitura de volta e reconciliação com a fonte

**Passos desta seção:** ler o Parquet recém-gravado diretamente do bucket e comparar a contagem de linhas com a fonte no BigQuery.

🎓 **Conceito, reconciliação** (*ETL Pipelines, Aula 1*): a validação clássica da etapa de extração é a comparação de contagens entre origem e destino (a aula cita contagens e checksums como controles de completude). Se fonte e Bronze coincidem, a cópia é fiel; qualquer diferença exige investigação antes de prosseguir. Este é o ponto de partida do script de reconciliação que a pipeline de produção terá.

In [23]:
# --- 5.1 Ler de volta do lake e reconciliar com a fonte ---
import pandas as pd

# Ler o Parquet DIRETO do bucket (round-trip completo: escrevi -> leio)
df_lido = pd.read_parquet(caminho, storage_options={"token": credenciais})

# Contagem na fonte (COUNT(*) é barato: nao traz os dados, so o numero)
qtd_fonte = pandas_gbq.read_gbq(
    f"SELECT COUNT(*) AS n FROM `{FONTE}.{TABELA}`",
    project_id=PROJETO_GCP, progress_bar_type=None,
)["n"][0]

print(f"Linhas na fonte (BigQuery):  {qtd_fonte}")
print(f"Linhas na Bronze (Parquet):  {len(df_lido)}")
print(f"Colunas na Bronze:           {len(df_lido.columns)}  (15 da fonte + 2 metadados)")
print()
if len(df_lido) == qtd_fonte:
    print("RECONCILIACAO OK — a Bronze é cópia fiel da fonte ✔")
else:
    print("DIVERGENCIA — investigar antes de prosseguir!")

df_lido.head(3)

Linhas na fonte (BigQuery):  145
Linhas na Bronze (Parquet):  145
Colunas na Bronze:           18  (15 da fonte + 2 metadados)

RECONCILIACAO OK — a Bronze é cópia fiel da fonte ✔


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source,data_ingestao
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-09T02:40:01.242047+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf,2026-07-09
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-09T02:40:01.242047+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf,2026-07-09
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-09T02:40:01.242047+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf,2026-07-09


## 6. Idempotência: reexecução não pode duplicar

**Passos desta seção:** executar a mesma gravação novamente e verificar que a Bronze não acumulou dados repetidos.

🎓 **Conceito, idempotência** (*ETL Pipelines, Aula 1*): pipelines reais são reexecutadas com frequência (falhas no meio do processo, reexecuções agendadas, correções manuais). Uma ingestão idempotente pode ser executada N vezes com o mesmo resultado de uma execução única. A estratégia adotada é a sobrescrita da partição: o caminho de gravação é determinístico (`bronze/uf/data_ingestao=<dia>/uf.parquet`), portanto reexecutar no mesmo dia substitui o arquivo em vez de acumular.

⚠️ **Nuance importante:** partições de dias diferentes são desejadas: constituem o histórico de cargas da Bronze (auditoria), não duplicação. Duplicação seria a existência de dois arquivos com os mesmos dados na mesma partição, e é isso que o teste verifica.

In [24]:
# --- 6.1 Segunda gravação + prova de não-duplicação ---
# Executa a MESMA gravação da célula 4.2 (simulando uma reexecução)
df_uf.to_parquet(caminho, index=False, storage_options={"token": credenciais})
print("Segunda gravação executada (mesmo caminho da primeira)")
print()

# Verificação 1: quantos arquivos existem NA PARTICAO DE HOJE?
particao_hoje = f"bronze/{TABELA}/data_ingestao={DATA_INGESTAO}/"
arquivos = list(cliente_storage.list_blobs(BUCKET_LAKE, prefix=particao_hoje))
print(f"Arquivos na partição de hoje ({particao_hoje}):")
for b in arquivos:
    print(f"  {b.name}  ({b.size:,} bytes)")

# Verificação 2: a contagem de linhas continua a mesma?
df_relido = pd.read_parquet(caminho, storage_options={"token": credenciais})
print()
print(f"Linhas após a 2ª gravação: {len(df_relido)}  (esperado: {len(df_uf)})")
print()
if len(arquivos) == 1 and len(df_relido) == len(df_uf):
    print("IDEMPOTENCIA OK — rodar 2x manteve 1 arquivo e a mesma contagem ✔")
else:
    print("PROBLEMA — algo acumulou; investigar a estratégia de sobrescrita!")

Segunda gravação executada (mesmo caminho da primeira)

Arquivos na partição de hoje (bronze/uf/data_ingestao=2026-07-09/):
  bronze/uf/data_ingestao=2026-07-09/uf.parquet  (17,883 bytes)

Linhas após a 2ª gravação: 145  (esperado: 145)

IDEMPOTENCIA OK — rodar 2x manteve 1 arquivo e a mesma contagem ✔


## 7. Generalização: o ciclo validado torna-se função

**Passos desta seção:** (7.1) empacotar o ciclo extrair, carimbar, gravar e reconciliar em uma função; (7.2) executá-la para todas as tabelas batch e emitir o relatório da carga.

🎓 **Conceito, de células para função:** o ciclo foi validado passo a passo nas Seções 4 a 6; repetir aquelas células para cada tabela multiplicaria pontos de erro. A função `ingerir_tabela()` é a unidade de produção: é ela que será promovida para `src/ingestion/prod_ingestao_batch.py` ao final da etapa.

🎓 **Conceito, relatório de carga** (*ETL Pipelines, Aula 1, e Etapa 7 do roadmap*): a função devolve métricas (linhas, reconciliação, destino) em vez de apenas imprimir. Essa lista de resultados é a base do monitoramento: volume processado por tabela por execução, uma das observabilidades solicitadas pelo enunciado.

ℹ️ **Observação:** a tabela `uf` será reingerida pelo loop, intencionalmente: a idempotência demonstrada na Seção 6 garante que não há efeito colateral, e o loop permanece uniforme.

In [25]:
# --- 7.1 O ciclo validado vira uma função ---
def ingerir_tabela(tabela: str) -> dict:
    """Ingere uma tabela da fonte para a Bronze e devolve as métricas da carga."""
    # 1. EXTRAIR (full) da fonte — Seção 4.1
    df = pandas_gbq.read_gbq(
        f"SELECT * FROM `{FONTE}.{tabela}`",
        project_id=PROJETO_GCP, progress_bar_type=None,
    )

    # 2. CARIMBAR os metadados de ingestão — Seção 4.1
    momento = datetime.now(timezone.utc)
    df["_ingestion_timestamp"] = momento.isoformat()
    df["_source"] = f"{FONTE}.{tabela}"

    # 3. GRAVAR Parquet na partição do dia (sobrescrita = idempotência) — Seções 4.2 e 6
    dia = momento.strftime("%Y-%m-%d")
    destino = f"gs://{BUCKET_LAKE}/bronze/{tabela}/data_ingestao={dia}/{tabela}.parquet"
    df.to_parquet(destino, index=False, storage_options={"token": credenciais})

    # 4. RECONCILIAR contagens fonte × Bronze — Seção 5
    qtd_fonte = pandas_gbq.read_gbq(
        f"SELECT COUNT(*) AS n FROM `{FONTE}.{tabela}`",
        project_id=PROJETO_GCP, progress_bar_type=None,
    )["n"][0]

    return {
        "tabela":        tabela,
        "linhas":        len(df),
        "fonte":         int(qtd_fonte),
        "reconciliacao": "OK" if len(df) == qtd_fonte else "DIVERGIU",
        "destino":       destino,
    }

print("Função ingerir_tabela() definida ✔")

Função ingerir_tabela() definida ✔


In [26]:
# --- 7.2 Escalar: rodar para todas as tabelas batch + relatório da carga ---
resultados = []
for t in TABELAS_BATCH:
    r = ingerir_tabela(t)
    resultados.append(r)
    print(f"  {r['tabela']:<32} {r['linhas']:>10,} linhas   reconciliação: {r['reconciliacao']}")

print()
divergentes = [r for r in resultados if r["reconciliacao"] != "OK"]
if not divergentes:
    print(f"BRONZE BATCH COMPLETA — {len(resultados)} tabelas ingeridas e reconciliadas ✔")
    print("(faltando apenas a gigante: alunos — Seção 8)")
else:
    print(f"ATENÇÃO: {len(divergentes)} tabela(s) divergiram — investigar antes de seguir!")

  uf                                      145 linhas   reconciliação: OK
  municipio                            23,995 linhas   reconciliação: OK
  meta_alfabetizacao_brasil                 3 linhas   reconciliação: OK
  meta_alfabetizacao_uf                    81 linhas   reconciliação: OK
  meta_alfabetizacao_municipio         10,704 linhas   reconciliação: OK
  dicionario                               27 linhas   reconciliação: OK

BRONZE BATCH COMPLETA — 6 tabelas ingeridas e reconciliadas ✔
(faltando apenas a gigante: alunos — Seção 8)


## 8. Ingestão dos microdados de `alunos`

**Passos desta seção:** (8.1) ingerir a maior tabela da fonte (3,9 milhões de linhas) com a mesma função da Seção 7, medindo a duração; (8.2) comparar o tamanho em Parquet com o equivalente estimado em CSV.

🎓 **Conceito, generalização em escala:** se o desenho estiver correto, escalar de 145 para 3,9 milhões de linhas não altera o código, apenas o tempo de execução. É o teste final da função antes da promoção para `src/`.

🎓 **Conceito, FinOps em escala** (*ETL Pipelines, Aula 4*): na tabela `uf` (145 linhas), Parquet e CSV apresentaram tamanhos próximos, porque o overhead de metadados do formato domina em tabelas pequenas. Com volume, a compressão colunar reduz o tamanho de forma significativa; esta seção mede a diferença com os dados do projeto.

⚙️ **Dependência de performance:** a extração utiliza a biblioteca `google-cloud-bigquery-storage`, que baixa os dados pela API de alta velocidade do BigQuery. Sem ela, o download de 3,9 milhões de linhas usa a API REST e leva dezenas de minutos. A dependência entrará no `requirements.txt` na promoção para produção. Observação: bibliotecas instaladas com o kernel em execução exigem reinício do kernel para serem reconhecidas.

In [27]:
# --- 8.1 Ingerir alunos com a MESMA função (medindo a duração) ---
import time

t0 = time.time()
r_alunos = ingerir_tabela("alunos")     # mesma função da Seção 7, sem alterações
duracao = time.time() - t0

print(f"  {r_alunos['tabela']:<32} {r_alunos['linhas']:>10,} linhas   reconciliação: {r_alunos['reconciliacao']}")
print(f"  duração da carga: {duracao/60:.1f} min")
print()
if r_alunos["reconciliacao"] == "OK":
    print("BRONZE BATCH 100% COMPLETA — as 7 tabelas da fonte estão no lake ✔")

  alunos                            3,867,999 linhas   reconciliação: OK
  duração da carga: 12.6 min

BRONZE BATCH 100% COMPLETA — as 7 tabelas da fonte estão no lake ✔


In [28]:
# --- 8.2 Comparação de tamanho: CSV × Parquet em escala ---
# Tamanho REAL do Parquet de alunos gravado no lake:
blobs_alunos = list(cliente_storage.list_blobs(BUCKET_LAKE, prefix="bronze/alunos/"))
tam_parquet = sum(b.size for b in blobs_alunos)

# Tamanho em CSV: ESTIMADO por amostragem (materializar 3,9 mi de linhas
# como texto na memória seria pesado à toa — medimos a média de bytes
# por linha numa amostra e extrapolamos)
AMOSTRA = 50_000
df_amostra = pandas_gbq.read_gbq(
    f"SELECT * FROM `{FONTE}.alunos` LIMIT {AMOSTRA}",
    project_id=PROJETO_GCP, progress_bar_type=None,
)
bytes_por_linha = len(df_amostra.to_csv(index=False).encode("utf-8")) / len(df_amostra)
tam_csv_estimado = int(bytes_por_linha * r_alunos["linhas"])

print(f"Parquet real no lake:  {tam_parquet/1024/1024:>8.1f} MB")
print(f"CSV estimado:          {tam_csv_estimado/1024/1024:>8.1f} MB   (extrapolado de {AMOSTRA:,} linhas)")
print()
print(f"Em escala, o Parquet ficou {tam_csv_estimado/tam_parquet:.1f}x menor que o CSV")
print("(na tabela uf, com 145 linhas, a razão foi de aproximadamente 1,1x)")

Parquet real no lake:     135.3 MB
CSV estimado:             205.4 MB   (extrapolado de 50,000 linhas)

Em escala, o Parquet ficou 1.5x menor que o CSV
(na tabela uf, com 145 linhas, a razão foi de aproximadamente 1,1x)


## 9. Consulta de validação: leitura direta da Bronze

**Passos desta seção:** consultar a camada Bronze diretamente do lake, sem passar pelo
BigQuery, respondendo uma pergunta de negócio simples (maiores taxas de alfabetização
por UF em 2024, rede pública).

🎓 **Conceito:** esta consulta demonstra que o lake é autossuficiente como fonte
analítica. A investigação também revelou características da coluna `rede` na tabela
`uf` (apenas os códigos 2, 3 e 5 estão presentes), registradas para a decisão de
rede de análise no diário de decisões.

In [30]:
#teste

# --- Teste de consulta direto na Bronze (sem passar pelo BigQuery) ---
# Lê o Parquet do lake e responde uma pergunta simples:
# quais UFs tiveram a maior taxa de alfabetização em 2024, na rede pública?

df_bronze_uf = pd.read_parquet(
    f"gs://{BUCKET_LAKE}/bronze/uf/data_ingestao={DATA_INGESTAO}/uf.parquet",
    storage_options={"token": credenciais},
)

(df_bronze_uf[(df_bronze_uf["ano"] == 2024) & (df_bronze_uf["rede"] == "5")]
    .nlargest(5, "taxa_alfabetizacao")
    [["ano", "sigla_uf", "rede", "taxa_alfabetizacao", "_ingestion_timestamp"]]
)

,ano,sigla_uf,rede,taxa_alfabetizacao,_ingestion_timestamp
76,2024,CE,5,85.31,2026-07-09T02:40:15.089479+00:00
85,2024,GO,5,72.74,2026-07-09T02:40:15.089479+00:00
82,2024,MG,5,72.07,2026-07-09T02:40:15.089479+00:00
78,2024,ES,5,71.69,2026-07-09T02:40:15.089479+00:00
87,2024,PR,5,70.42,2026-07-09T02:40:15.089479+00:00
